In [13]:
import numpy as np

# Regenerate inventory_snapshots.csv with realistic stock levels
# (fixes the original generator's downward-drift bug that left
# stock chronically too low relative to demand)
rng = np.random.default_rng(42)

calendar_df = pd.read_csv("../data/raw/calendar.csv", parse_dates=["date"])
sku_ids = pd.read_csv("../data/raw/sku_master.csv")["sku_id"].unique()
snap_dates = calendar_df[calendar_df["date"].dt.dayofweek == 0]["date"]

rows = []
for sku_id in sku_ids:
    on_hand = rng.integers(80, 300)
    lead_time = int(rng.integers(7, 30))
    reorder_point = int(rng.integers(15, 60))
    for d in snap_dates:
        weekly_draw = rng.integers(5, 35)
        on_hand = max(0, on_hand - weekly_draw)
        if on_hand <= reorder_point and rng.random() < 0.6:
            restock = rng.integers(80, 250)
            on_order = restock
            on_hand += restock
        else:
            on_order = rng.integers(0, 30) if rng.random() < 0.2 else 0
        rows.append([d, sku_id, on_hand, on_order, lead_time, reorder_point])

fixed_inventory = pd.DataFrame(rows, columns=["date", "sku_id", "on_hand_units",
                                                "on_order_units", "lead_time_days", "reorder_point"])
fixed_inventory.to_csv("../data/raw/inventory_snapshots.csv", index=False)
fixed_inventory.to_csv("../data/processed/inventory_snapshots_clean.csv", index=False)

print("Inventory data regenerated and saved.")
fixed_inventory["on_hand_units"].describe()

Inventory data regenerated and saved.


count    21000.000000
mean       115.846524
std         62.960093
min          0.000000
25%         67.000000
50%        110.000000
75%        161.000000
max        299.000000
Name: on_hand_units, dtype: float64

In [14]:


import pandas as pd

inventory = pd.read_csv("../data/processed/inventory_snapshots_clean.csv",parse_dates=["date"])

print(f"Rows: {len(inventory)}")
inventory.head()


Rows: 21000


,date,sku_id,on_hand_units,on_order_units,lead_time_days,reorder_point
0,2024-01-01,SKU0001,81,0,24,44
1,2024-01-08,SKU0001,74,20,24,44
2,2024-01-15,SKU0001,54,0,24,44
3,2024-01-22,SKU0001,20,25,24,44
4,2024-01-29,SKU0001,113,111,24,44


In [15]:
# Use the average of the last 4 weeks instead of a single snapshot —
# smooths out any one unusually low week rather than treating it as
# the permanent current stock level
recent_inventory = inventory.sort_values("date").groupby("sku_id").tail(4)
latest_inventory = recent_inventory.groupby("sku_id").agg({
    "on_hand_units": "mean",
    "on_order_units": "mean",
    "lead_time_days": "last",
    "reorder_point": "last"
}).reset_index()

latest_inventory["on_hand_units"] = latest_inventory["on_hand_units"].round().astype(int)
latest_inventory["on_order_units"] = latest_inventory["on_order_units"].round().astype(int)

print(f"Products: {len(latest_inventory)}")
latest_inventory["on_hand_units"].describe()


Products: 200


count    200.000000
mean     115.700000
std       50.819021
min        2.000000
25%       81.750000
50%      106.500000
75%      150.000000
max      261.000000
Name: on_hand_units, dtype: float64

In [16]:
# Get code each products latest forecast
# Bring back the 200-sku weekly data( same as forecast notebook)
df = pd.read_csv("../data/processed/analysis_ready.csv",parse_dates=['date'])

all_weekly = df.set_index("date").groupby("sku_id").resample("W")["units_sold"].sum().reset_index()
all_weekly = all_weekly.sort_values(["sku_id","date"]).reset_index(drop=True)

# Simple, transparent expected-weekly-demand estimate: each SKU's average
# of its last 8 weeks (recent, stable signal - easy to explain to a non-technical stakeholder which the breif requiresfor this layer )
recent_avg = all_weekly.groupby("sku_id").apply(
    lambda x: x.tail(8)["units_sold"].mean()
).reset_index()
recent_avg.columns =["sku_id","expected_weekly_demand"]

recent_avg.head()


C:\Users\Shreyas\AppData\Local\Temp\ipykernel_14784\4237036656.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  recent_avg = all_weekly.groupby("sku_id").apply(


,sku_id,expected_weekly_demand
0,SKU0001,31.250
1,SKU0002,5.375
2,SKU0003,33.000
3,SKU0004,99.000
4,SKU0005,104.750


In [17]:
sku_master = pd.read_csv("../data/raw/sku_master.csv")

risk_df = latest_inventory.merge(recent_avg, on="sku_id", how="left")
risk_df = risk_df.merge(sku_master[["sku_id", "category", "list_price"]], on="sku_id", how="left")

risk_df.head()

,sku_id,on_hand_units,on_order_units,lead_time_days,reorder_point,expected_weekly_demand,category,list_price
0,SKU0001,232,1,24,44,31.250,Furniture,289.67
1,SKU0002,113,25,24,53,5.375,Decor,338.84
2,SKU0003,103,0,9,36,33.000,Appliances,129.72
3,SKU0004,115,45,26,32,99.000,Furniture,234.82
4,SKU0005,138,46,24,19,104.750,Appliances,126.11


In [18]:
# How much we'll likely sell during the time it takes to restock
risk_df["demand_over_lead_time"] = risk_df["expected_weekly_demand"] * (risk_df["lead_time_days"] / 7)

# Total stock available (what's physically here + what's coming)
risk_df["total_available"] = risk_df["on_hand_units"] + risk_df["on_order_units"]

# Stockout risk score: how much of the lead-time demand is NOT covered by stock
# 0 = fully covered (safe), 1+ = demand exceeds available stock (danger)
risk_df["stockout_risk"] = (risk_df["demand_over_lead_time"] - risk_df["total_available"]) / risk_df["demand_over_lead_time"]
risk_df["stockout_risk"] = risk_df["stockout_risk"].clip(lower=0, upper=1)

risk_df[["sku_id", "expected_weekly_demand", "lead_time_days", "demand_over_lead_time",
         "total_available", "stockout_risk"]].sort_values("stockout_risk", ascending=False).head(10)

,sku_id,expected_weekly_demand,lead_time_days,demand_over_lead_time,total_available,stockout_risk
116,SKU0114,222.875,19,604.946429,16,0.973551
19,SKU0020,253.250,27,976.821429,28,0.971336
101,SKU0099,205.625,24,705.000000,27,0.961702
141,SKU0139,102.000,21,306.000000,15,0.950980
29,SKU0030,265.375,27,1023.589286,79,0.922821
178,SKU0176,274.125,10,391.607143,36,0.908071
10,SKU0011,184.875,22,581.035714,54,0.907063
154,SKU0152,84.750,24,290.571429,28,0.903638
48,SKU0047,245.000,24,840.000000,98,0.883333
135,SKU0133,263.625,18,677.892857,98,0.855434


In [19]:
# Forward window: how much we expect to sell in the next 8 weeks (2 months)
forward_weeks = 8
risk_df["demand_next_8weeks"] = risk_df["expected_weekly_demand"] * forward_weeks

# Overstock risk score: how much MORE stock we have than we'll sell in that window
# 0 = stock roughly matches demand (healthy), 1+ = holding way more than needed
risk_df["overstock_risk"] = (risk_df["on_hand_units"] - risk_df["demand_next_8weeks"]) / risk_df["on_hand_units"]
risk_df["overstock_risk"] = risk_df["overstock_risk"].clip(lower=0, upper=1)

risk_df[["sku_id", "on_hand_units", "expected_weekly_demand", "demand_next_8weeks", "overstock_risk"]] \
    .sort_values("overstock_risk", ascending=False).head(10)

,sku_id,on_hand_units,expected_weekly_demand,demand_next_8weeks,overstock_risk
66,SKU0064,180,3.375,27.0,0.850000
136,SKU0134,156,3.000,24.0,0.846154
140,SKU0138,185,4.750,38.0,0.794595
47,SKU0046,194,5.750,46.0,0.762887
119,SKU0117,156,4.750,38.0,0.756410
72,SKU0070,108,3.500,28.0,0.740741
166,SKU0164,137,4.500,36.0,0.737226
54,SKU0053,152,5.000,40.0,0.736842
11,SKU0012,147,4.875,39.0,0.734694
93,SKU0091,180,6.250,50.0,0.722222


In [20]:
import numpy as np

# Can't be overstocked if you have zero stock on hand — fix the divide-by-zero
risk_df["overstock_risk"] = np.where(
    risk_df["on_hand_units"] == 0,
    0,
    (risk_df["on_hand_units"] - risk_df["demand_next_8weeks"]) / risk_df["on_hand_units"]
)
risk_df["overstock_risk"] = risk_df["overstock_risk"].clip(lower=0, upper=1)

risk_df[["sku_id", "on_hand_units", "expected_weekly_demand", "demand_next_8weeks", "overstock_risk"]] \
    .sort_values("overstock_risk", ascending=False).head(10)

,sku_id,on_hand_units,expected_weekly_demand,demand_next_8weeks,overstock_risk
66,SKU0064,180,3.375,27.0,0.850000
136,SKU0134,156,3.000,24.0,0.846154
140,SKU0138,185,4.750,38.0,0.794595
47,SKU0046,194,5.750,46.0,0.762887
119,SKU0117,156,4.750,38.0,0.756410
72,SKU0070,108,3.500,28.0,0.740741
166,SKU0164,137,4.500,36.0,0.737226
54,SKU0053,152,5.000,40.0,0.736842
11,SKU0012,147,4.875,39.0,0.734694
93,SKU0091,180,6.250,50.0,0.722222


In [21]:
risk_df[["sku_id", "on_hand_units", "overstock_risk"]].sort_values("overstock_risk", ascending=False).head(10)

,sku_id,on_hand_units,overstock_risk
66,SKU0064,180,0.850000
136,SKU0134,156,0.846154
140,SKU0138,185,0.794595
47,SKU0046,194,0.762887
119,SKU0117,156,0.756410
72,SKU0070,108,0.740741
166,SKU0164,137,0.737226
54,SKU0053,152,0.736842
11,SKU0012,147,0.734694
93,SKU0091,180,0.722222


In [22]:
def classify_risk(row, threshold=0.5):
    high_stockout = row["stockout_risk"] >= threshold
    high_overstock = row["overstock_risk"] >= threshold

    if high_stockout and high_overstock:
        return "Watch / Volatile"
    elif high_stockout:
        return "Reorder Now"
    elif high_overstock:
        return "Markdown / Clear"
    else:
        return "Healthy"

risk_df["risk_zone"] = risk_df.apply(classify_risk, axis=1)

risk_df["risk_zone"].value_counts()

risk_zone
Healthy             109
Reorder Now          71
Markdown / Clear     23
Name: count, dtype: int64

In [23]:
risk_df["stockout_risk"].describe()

count    203.000000
mean       0.321904
std        0.331642
min        0.000000
25%        0.000000
50%        0.276046
75%        0.612153
max        0.973551
Name: stockout_risk, dtype: float64

In [24]:
risk_df[["sku_id", "expected_weekly_demand", "lead_time_days", "demand_over_lead_time",
         "on_hand_units", "on_order_units", "total_available", "stockout_risk"]].head(10)

,sku_id,expected_weekly_demand,lead_time_days,demand_over_lead_time,on_hand_units,on_order_units,total_available,stockout_risk
0,SKU0001,31.250,24,107.142857,232,1,233,0.000000
1,SKU0002,5.375,24,18.428571,113,25,138,0.000000
2,SKU0003,33.000,9,42.428571,103,0,103,0.000000
3,SKU0004,99.000,26,367.714286,115,45,160,0.564880
4,SKU0005,104.750,24,359.142857,138,46,184,0.487669
5,SKU0006,207.500,18,533.571429,86,0,86,0.838822
6,SKU0007,30.875,27,119.089286,199,1,200,0.000000
7,SKU0008,5.500,8,6.285714,136,46,182,0.000000
8,SKU0009,103.750,12,177.857143,106,4,110,0.381526
9,SKU0010,92.500,27,356.785714,104,4,108,0.697297


In [25]:
risk_df["risk_zone"].value_counts()

risk_zone
Healthy             109
Reorder Now          71
Markdown / Clear     23
Name: count, dtype: int64

In [29]:
# Calculate rupee impact
# Money at risk for stockout-flagged SKUs: lost sales value
risk_df["stockout_value_at_risk"] = np.where(
    risk_df["risk_zone"]=="Reorder Now",
    risk_df["expected_weekly_demand"] * risk_df["list_price"],
    0
)

# Money locked up in overstock-flagged SKUs: cost of excess inventory
risk_df["overstock_value_at_risk"] = np.where(
    risk_df["risk_zone"] == "Markdown / Clear",
    risk_df["on_hand_units"]* risk_df["list_price"],
    0 
) 

total_stockout_risk_value = risk_df["stockout_value_at_risk"].sum()
total_overstock_value = risk_df["overstock_value_at_risk"].sum()

print(f"Total revenue at risk from stockouts (next week): ${total_stockout_risk_value:,.0f}")
print(f"Total capital locked in overstock: ${total_overstock_value:,.0f}")



Total revenue at risk from stockouts (next week): $1,743,565
Total capital locked in overstock: $539,328


In [30]:
risk_df.to_csv("../data/processed/risk_scores.csv", index="False")
print("Saved risk_scores.csv with", len(risk_df) , "rows")


Saved risk_scores.csv with 203 rows
